# Scrape CSD and curate the synonyms
NOTE: you need a CSD licence to install and use the CSD API

In [1]:
from pathlib import Path
from tqdm import tqdm
import pandas as pd

In [3]:
import ccdc
import ccdc.io
import ccdc.search
print("ccdc.__version__", ccdc.__version__)
print("ccdc.io.csd_directory()", ccdc.io.csd_directory())
print("ccdc.io.csd_version()", ccdc.io.csd_version())

subset_mofs_all = Path(ccdc.io.csd_directory()) / "subsets" / "CSD_MOF_subsets" / "MOF_subset.gcd"
print(f"Number of MOFs: {len(ccdc.io.EntryReader(str(subset_mofs_all)))}")

ccdc.__version__ 3.6.1
ccdc.io.csd_directory() C:/Users/james/CCDC/ccdc-data/csd
ccdc.io.csd_version() 601
Number of MOFs: 135253


In [4]:
n_entries = len(ccdc.io.EntryReader(str(subset_mofs_all)))
rows = [{} for _ in range(n_entries)] # Pre allocate for speed
for i, entry in enumerate(tqdm(ccdc.io.EntryReader(str(subset_mofs_all)))): # Takes 2 minutes
    for col in ['identifier', 'ccdc_number','formula', 'has_disorder']:
        rows[i][col] = getattr(entry, col)
    rows[i]['publication_year'] = entry.publication.year
    doi = entry.publication.doi
    if doi: # can be None
        doi = doi.lower() # Lower caption to improve match in a later step
    rows[i]['publication_doi'] = doi 
    synonyms = list(entry.synonyms)
    rows[i]['synonyms_orig'] = synonyms
df = pd.DataFrame(rows) 

df.to_csv("data/step-00.csv", index=False) 
# step-00.csv contains all CSD entries of MOF subset: id, formula, has_disorder, pub year, doi, synonyms_orig

100%|██████████| 135253/135253 [09:17<00:00, 242.68it/s]


In [6]:
df.head(3)

,identifier,ccdc_number,formula,has_disorder,publication_year,publication_doi,synonyms_orig
0,ABACUF,1100034,(C6 H14 Ba2 Cu1 O16)n,False,1958,None,[]
1,ABACUF01,230290,(C6 H14 Ba2 Cu1 O16)n,False,2004,10.1016/j.molstruc.2004.03.051,[]
2,ABADUG,2104232,"2(C51 H30 N2 O9 Zn2)n,(C51 H30 N2 O9 Zn2)n",True,2021,None,[]


In [7]:
df = pd.read_csv("data/step-00.csv")
print("CSD entries having synonyms:", len(df[df['synonyms_orig']!="[]"]))

CSD entries having synonyms: 12922


In [8]:
# Correct the names from "synonyms_orig", and create a new column "synonyms": exclude misleading ones, and solvents/adsorbate.
# NOTE: the reason why I'm doing it here and not later in the name match, is to exclude those names that are obviously problematic,
#       and I want to remove completely, to make later checks easier.

ADSORBATES_LIST = [
    "carbon", # carbon dioxide
    "deuteromethane",
    "methane",
    "ethane",
    "ethene",
    "dinitrogen",
    "acetylene",
    "acetylene",
    "acetaldehyde",
    "ethylamine"  
]

df = pd.read_csv("data/step-00.csv")
df['synonyms'] = None # create new column for .at
for i, synonyms in enumerate(tqdm(df['synonyms_orig'])):
    synonyms_valid = [] 
    for name in eval(synonyms): #there may be more than one synonym, but it is not very frequent
        # Exclude some fake names:
        if any([
            name.startswith('catena'),
            name.startswith('Teaching Subset'),
            name.startswith('DrugBank')
            #len(name)>20
        ]):
            continue
            
        if name in ["","MOF-1", "MOF-2", 1, 2, "1", "2"]:
            continue
            
        name_split = name.split(" ")
        if len(name_split)>1 and name_split[1] in ADSORBATES_LIST: # remove adsorbent from name
            name = name_split[0] 
            
        synonyms_valid.append(name)
        
    df.at[i,'synonyms'] = synonyms_valid

df.to_csv("data/step-01.csv", index=False) 
# step-01.csv: add "synonyms" column with corrected CSD syns

100%|██████████| 135253/135253 [00:04<00:00, 27065.67it/s]


In [9]:
df = pd.read_csv("data/step-01.csv")
print("CSD entries having valid synonyms:", len(df[df['synonyms']!="[]"]))

CSD entries having valid synonyms: 9410


In [10]:
df = pd.read_csv("data/step-01.csv")
df[df['synonyms']!="[]"].head(3)

,identifier,ccdc_number,formula,has_disorder,publication_year,publication_doi,synonyms_orig,synonyms
13,ABAVIM01,1984147,"(C24 H26 N3 Ni1 O11 P3)n,H2 O1",True,2021,10.1007/s12274-020-3056-6,['IEF-13'],['IEF-13']
37,ABECIX,2081306,(C52 H34 Ag1 Co3 F1 N2 O13 P2)n,True,2021,10.1021/jacs.1c05564,['Ag-PCM-102'],['Ag-PCM-102']
38,ABECOD,2081307,"0.71(C55 H43 Ag1 Co3 F1 N3 O15 P2)n,0.29(C55 H...",True,2021,10.1021/jacs.1c05564,['AgCu-PCM-102'],['AgCu-PCM-102']


In [11]:
# Inspect names that have a space: one can see that many more can be fixed but it is hard to find general rules
df = pd.read_csv("data/step-01.csv")
for i in df.index:
    synonyms = eval(df.at[i,'synonyms'])
    if len(synonyms)>0:
        first = synonyms[0]
        if len(first.split(" "))>1:
            print(first)

C32 H42 O20 S4 Zn4
MAF-40 unknown solvate
CSUST-4 N,N-dimethylformamide solvate
di-ammonium catena-(diuranyl trioxalate)
MFM-303(Al) ammonia
MFM-303(Al) hydrate
Anhydrous sodium naproxen
IRH-5-as synthesised
IRH-4-as synthesised
amoxicillin sodium
amoxicillin sodium methyl acetate solvate
copper-doped UiO-66 unknown solvate
UiO-66 unknown solvate
EMOF 1
ZUL-330 /  ZUL-430 acetylene
ZUL-430 activated
ZUL-330 activated
IHEP-17 unknown solvate
IHEP-17-I unknown solvate
IHEP-18 unknown solvate
Sodium Acetate
Sodium Acetate
Sodium acetate
ZUL-C5 2,2-dimethylbutane
ZUL-C5 n-hexane solvate
ZUL-C5 hemikis(3-methylpentane)
Strontium Manganese Trifluoroacetate
activated NTU-101-NH2
activated NTU-101
MIL-100 (Al)
NU-400 MOF
Uranyl phenylphosphinate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 octadecahydrate
MOF-303 hydrate
MOF-303 hydrate
MOF-303 hexadecahydrate
MOF-303 hydr

In [12]:
# Check papers with a lot of entries
doi_note = {
    "10.1002/cssc.201601752": "in-situ study of M-MOF-74",
}

for doi, note in doi_note.items():
    df_doi = df[df['publication_doi']==doi]
    print(doi, f"({note})", f"Number of CIFs: {len(df_doi)}", 'Synonyms:', *list(set(df_doi['synonyms'])))

10.1002/cssc.201601752 (in-situ study of M-MOF-74) Number of CIFs: 1853 Synonyms: ['CPO-27-Mn'] ['CPO-27-Co'] ['CPO-27-Cu'] ['CPO-27-Zn'] ['CPO-27-Mg'] ['CPO-27-Ni'] []


In [13]:
# Keep only 3 MOFs for the same DOI with the same synonyms, to avoid having paper with hundreds of structures
df = pd.read_csv("data/step-01.csv")
df['note'] = "-" # Assign a string for all the notes, to make the later filtering easier
df = df.sort_values(by=['publication_doi','synonyms','identifier'])
df = df.reset_index() # keeping the old as column "index"

max_same = 3
count_same=0
for i in tqdm(df.index[1:]): # skipping the first because it is comparing with the previous (no privious for the first!)
    if df.at[i,'synonyms']!='[]':
        if df.at[i,'publication_doi']==df.at[i-1,'publication_doi'] and df.at[i,'synonyms']==df.at[i-1,'synonyms']:
            count_same+=1
        else:
            count_same=0
        if count_same>=3:
            df.at[i, 'note'] = f"Excluding more than {max_same} same MOFs from the same DOI"

df = df.sort_values(by='index')
df = df.drop(columns='index') # remove the dummy column
df.to_csv("data/step-02.csv", index=False)
# step-02.csv: contains a "note" for excluding structures if >3 from the same paper

100%|██████████| 135252/135252 [00:01<00:00, 127828.32it/s]


In [14]:
df = pd.read_csv("data/step-02.csv")
len_after_exclusions = len(df[(df['synonyms']!="[]") & (df['note']=='-')])
print("CSD entries having synonyms:", len_after_exclusions, f"(including only {max_same} same materials per paper)")

CSD entries having synonyms: 7043 (including only 3 same materials per paper)


In [15]:
df = pd.read_csv("data/step-02.csv")
df[df['synonyms']!="[]"].head(3)

,identifier,ccdc_number,formula,has_disorder,publication_year,publication_doi,synonyms_orig,synonyms,note
13,ABAVIM01,1984147,"(C24 H26 N3 Ni1 O11 P3)n,H2 O1",True,2021,10.1007/s12274-020-3056-6,['IEF-13'],['IEF-13'],-
37,ABECIX,2081306,(C52 H34 Ag1 Co3 F1 N2 O13 P2)n,True,2021,10.1021/jacs.1c05564,['Ag-PCM-102'],['Ag-PCM-102'],-
38,ABECOD,2081307,"0.71(C55 H43 Ag1 Co3 F1 N3 O15 P2)n,0.29(C55 H...",True,2021,10.1021/jacs.1c05564,['AgCu-PCM-102'],['AgCu-PCM-102'],-
